<a href="https://colab.research.google.com/github/OscarRojasG/Experimentos-EvaluAI/blob/main/Framework_EvaluAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Implementación

## Cargar dataset y librerías necesarias

In [1]:
!pip install openai==0.28 &> /dev/null
!pip install openai-multi-client &> /dev/null
!pip install plotly
!git clone https://github.com/rilianx/GPTEvaluator &> /dev/null

In [2]:
import pandas as pd
from IPython.display import display

# Muestra información relevante del dataset
def show_dataset_info(dataset):
    display(dataset.head())
    print()
    print(dataset.value_counts("real_eval"), end="\n\n")
    print(dataset.value_counts("dataset"))
    pass

# Carga un dataset a partir de un archivo xlsx y valida sus columnas
def load_dataset(path, sheet_name, column_data):
    df = pd.read_excel(path, sheet_name=sheet_name)

    mandatory_cols = ["context", "question", "answer", "real_eval", "dataset"]
    for key in mandatory_cols:
        if key not in column_data.keys():
            raise Exception(f"Error: Debe especificar la columna para la variable {key}")

        value = column_data[key]
        if value not in df.columns:
            raise Exception(f"Error: La columna {value} no existe")

        df = df.rename(columns={value: key})

    df = df[mandatory_cols]
    df['row'] = df.index + 2
    show_dataset_info(df)
    return df

## Generación prompts

In [3]:
import pprint
import copy
import json
import os
import re

class Prompt():
    def __init__(self, structure, instructions, base_folder):
        self.structure = structure
        self.instructions = instructions
        self.base_folder = base_folder
        self.raw_text_structure = None
        self.text_structure = None
        self.criteria = None
        self.output_instructions = None
        self.prompt = None

        self.read_files()
        self.extract_metadata()
        self.build_prompt()

    # Retorna la estructura base del prompt (diccionario)
    def base_structure(self):
        structure = copy.deepcopy(self.structure)
        structure['instructions'] = {}
        for i in self.instructions:
            structure['instructions'][i] = structure[i]
            structure.pop(i, None)
        return structure

    # Crea un diccionario con el contenido de cada archivo en la estructura
    def read_files(self):
        self.raw_text_structure = copy.deepcopy(self.structure)

        for key, value in self.raw_text_structure.items():
            if key == "instructions": continue

            path = f"{self.base_folder}/{key}/{value}"
            try:
                self.raw_text_structure[key] = open(path, 'r', encoding='utf-8').read()
                self.raw_text_structure[key] += "\n\n"
            except:
                raise Exception(f"Error: El archivo {path} no existe")

    # Extrae metadatos de los archivos como los criterios e instrucciones de salida
    def extract_metadata(self):
        self.text_structure = copy.deepcopy(self.raw_text_structure)
        self.output_instructions = {}
        self.criteria = []

        for key in self.text_structure:
            value = self.text_structure[key]

            if value.startswith('#') and not value.startswith('##'):
                m = re.search(r'#(.*?)\n', value).group(1)
                mkey, mvalue = m.split(":", 1)
                self.output_instructions[mkey.lstrip()] = mvalue.lstrip()
                self.text_structure[key] = '\n'.join(value.split('\n')[1:])

            if key == 'score':
                lines = self.text_structure['score'].split('\n')
                for line in lines:
                    if line.startswith('$'):
                        m = re.search(r'\$(.*?)\n', value).group(1)
                        mkey, mvalue = m.split(":", 1)

                        self.criteria.append(mkey.lstrip())
                        self.output_instructions[mkey.lstrip()] = mvalue.lstrip()
                        self.text_structure[key] = '\n'.join(value.split('\n')[1:])


    # Construye el prompt en formato string
    def build_prompt(self):
        self.prompt = ""
        for key, value in self.text_structure.items():
            self.prompt += value

        output = self.build_output()
        self.prompt += output

    def build_output(self):
        output = "I expect a dict in python as answer: {{"
        output += ', '.join(f'"{key}": \'{value}\'' for key, value in self.output_instructions.items())

        output += "}}\n\nPython dict:"
        return output


# Procesa y elimina los diccionarios anidados de prompt_data
def normalize_prompt_dict(prompt_data):
    instructions = []
    after_instructions = {}
    found_target = False

    if "instructions" in prompt_data:
        for key, value in prompt_data.items():
            if found_target:
                after_instructions[key] = value
            if key == "instructions":
                found_target = True

        for (key, value) in prompt_data["instructions"].items():
            prompt_data[key] = value
            instructions.append(key)

        for key, value in after_instructions.items():
            del prompt_data[key]
            prompt_data[key] = value

    prompt_data["instructions"] = "Instructions:\n"
    return prompt_data, instructions

# Retorna la lista de archivos para reemplazar el comodín *
def expand_prompt_data(prompt_data, prompt_folder):
    wildcard_field = None
    for key, value in prompt_data.items():
        if value == "*":
            wildcard_field = key
            break

    if not wildcard_field: return None, None

    wildcard_files = []
    path = f"{prompt_folder}/{wildcard_field}"
    for file in sorted(os.listdir(path)):
        if os.path.isfile(os.path.join(path, file)):
            wildcard_files.append(file)

    return wildcard_field, wildcard_files

# Genera una lista con los prompts a evaluar
def generate_prompts(prompt_data, prompt_folder, visualize=True):
    template, instructions = normalize_prompt_dict(prompt_data)
    wildcard_field, wildcard_files = expand_prompt_data(template, prompt_folder)
    prompts = []

    if wildcard_field == None:
        prompt = Prompt(template, instructions, prompt_folder)
        prompts.append(prompt)

        if visualize: print(prompt.prompt)
        return prompts

    for file in wildcard_files:
        structure = copy.deepcopy(template)
        structure[wildcard_field] = file
        prompt = Prompt(structure, instructions, prompt_folder)
        prompts.append(prompt)

    # Visualizar
    if visualize:
        template = copy.deepcopy(prompts[0])
        template.raw_text_structure[wildcard_field] = f"{{{wildcard_field}}}\n\n"
        template.extract_metadata()
        template.build_prompt()
        print(template.prompt)

        print(f"\n\nArchivos a utilizar ({len(wildcard_files)}):\n")
        print("\n".join(wildcard_files))

    return prompts

## Optimización parámetros

In [4]:
from scipy.optimize import differential_evolution
from abc import ABC, abstractmethod
import numpy as np

class ScoreWeighter():
    @staticmethod
    def eval(x, theta, right_offset):
        return np.dot(x, theta[:-right_offset])

class MapOptimizer(ABC):
    def __init__(self, map_params_size):
        self.map_params_size = map_params_size

    def optimize(self, criteria_scores, real_scores):
        bounds =  [(0, 1) for _ in range(len(criteria_scores[0]))] + [(0, 10)] * self.map_params_size
        result = differential_evolution(self.error, bounds, args=(criteria_scores, real_scores), seed=1, strategy='rand1exp', mutation=(0,1), recombination=1)
        return result.x.tolist()

    def error(self, theta, x, y):
        y_pred = self.f(x, theta)
        mse = np.sum((y - y_pred) ** 2)

        # Penalización cuando suma de ponderaciones != 1
        weights = theta[:-self.map_params_size]
        penalty = 1e6 * np.abs(np.sum(weights) - 1)

        # Penalización cuando parámetros de mapeo no están de menor a mayor
        map_params = theta[-self.map_params_size:]
        penalty += sum((map_params[i] - map_params[i+1]) * 1e5 for i in range(len(map_params)-1) if map_params[i] > map_params[i+1])
        return mse + penalty

    @abstractmethod
    def f(self, x, theta):
        pass

class MapOptimizer4(MapOptimizer):
    def __init__(self):
        super().__init__(4)

    def map_array(self, w_scores, theta):
        a, b, c, d = theta[-4:]
        def map(x):
            if x <= a:
                return 0
            if a < x <= b:
                return (x - a) / (b - a)
            if b < x <= c:
                return 1 + (x - b) / (c - b)
            if c < x <= d:
                return 2 + (x - c) / (d - c)
            else:
                return 3
        return np.array([map(x) for x in w_scores])

    def f(self, x, theta):
        w_scores = ScoreWeighter.eval(x, theta, 4)
        return self.map_array(w_scores, theta)

class MapOptimizer2(MapOptimizer):
    def __init__(self):
        super().__init__(2)

    def map_array(self, w_scores, theta):
        a, b = theta[-2:]
        def map(x):
            if x <= a:
                return x / a
            if a < x <= b:
                return 1 + (x - a) / (b - a)
            else:
                return 2 + (x - b) / (10 - b)
        return np.array([map(x) for x in w_scores])

    def f(self, x, theta):
        w_scores = ScoreWeighter.eval(x, theta, 2)
        return self.map_array(w_scores, theta)

class MapOptimizer2Simple(MapOptimizer):
    def __init__(self):
        super().__init__(2)

    def map_array(self, w_scores, theta, eps=1e-9):
        a, b = theta[-2:]
        def map(x):
            if x <= a:
                return x*eps
            if a < x <= b:
                return 3 * (x - a) / (b - a)
            else:
                return 3 + x*eps
        return np.array([map(x) for x in w_scores])

    def f(self, x, theta):
        w_scores = ScoreWeighter.eval(x, theta, 2)
        return self.map_array(w_scores, theta)

class MapOptimizer2Mini(MapOptimizer):
    def __init__(self):
        super().__init__(1)

    def map_array(self, w_scores, theta):
        a = theta[-1]
        def map(x):
            b = 10 - a
            if x <= a:
                return 0
            if a < x <= b:
                return 3 * (x - a) / (b - a)
            else:
                return 3
        return np.array([map(x) for x in w_scores])

    def f(self, x, theta):
        w_scores = ScoreWeighter.eval(x, theta, 1)
        return self.map_array(w_scores, theta)

class MapOptimizer4Mini(MapOptimizer):
    def __init__(self):
        super().__init__(3)

    def map_array(self, w_scores, theta):
        a, b, c = theta[-3:]
        def map(x):
            d = 10 - a
            if x <= a:
                return 0
            if a < x <= b:
                return (x - a) / (b - a)
            if b < x <= c:
                return 1 + (x - b) / (c - b)
            if c < x <= d:
                return 2 + (x - c) / (d - c)
            else:
                return 3
        return np.array([map(x) for x in w_scores])

    def f(self, x, theta):
        w_scores = ScoreWeighter.eval(x, theta, 3)
        return self.map_array(w_scores, theta)

# Obtiene los parámetros óptimos para disminuir el error
def optimize_params(criteria_scores, real_scores, eval_function):
    if eval_function == "map4":
        params = MapOptimizer4().optimize(criteria_scores, real_scores)
    if eval_function == "map4-mini":
        params = MapOptimizer4Mini().optimize(criteria_scores, real_scores)
    if eval_function == "map2" or eval_function == "map":
        params = MapOptimizer2().optimize(criteria_scores, real_scores)
    if eval_function == "map2-simple":
        params = MapOptimizer2Simple().optimize(criteria_scores, real_scores)
    if eval_function == "map2-mini":
        params = MapOptimizer2Mini().optimize(criteria_scores, real_scores)

    return params

# Obtiene los parámetros óptimos para disminuir el error
def convert_gpt_scores(criteria_scores, real_scores, eval_function, eval_params):
    if eval_function == "map4":
        return MapOptimizer4().f(criteria_scores, eval_params)
    if eval_function == "map4-mini":
        return MapOptimizer4Mini().f(criteria_scores, eval_params)
    if eval_function == "map2" or eval_function == "map":
        return MapOptimizer2().f(criteria_scores, eval_params)
    if eval_function == "map2-simple":
        return MapOptimizer2Simple().f(criteria_scores, eval_params)
    if eval_function == "map2-mini":
        return MapOptimizer2Mini().f(criteria_scores, eval_params)

In [6]:
criteria_scores = [[9], [8], [4]]
real_scores = [3, 2, 1]
eval_function = "map4"

eval_params = optimize_params(criteria_scores, real_scores, eval_function)
print(eval_params)
convert_gpt_scores(criteria_scores, real_scores, eval_function, eval_params)

[1.0, 1.1720268471946058, 3.9999999999999996, 8.0, 8.972909085620861]


array([3., 2., 1.])

In [8]:
criteria_scores = [[8, 9, 7], [8, 5, 3], [4, 1, 8]]
real_scores = [3, 2, 1]
eval_function = "map4"

eval_params = optimize_params(criteria_scores, real_scores, eval_function)
print(eval_params)
convert_gpt_scores(criteria_scores, real_scores, eval_function, eval_params)

[0.4287267016646835, 0.20566060222218852, 0.365612696113128, 1.941605522917902, 4.845468977785947, 5.5549547127677945, 7.3527984526670185]


array([3., 2., 1.])

## Experimentos

In [9]:
import random
from GPTEvaluator.GPTEvaluator import chat_gpt_multiple
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from openai_multi_client import OpenAIMultiClient
import openai
from google.colab import userdata
from sklearn.metrics import mean_squared_error
import os
from datetime import datetime, timedelta
import json
import numpy
import plotly.express as px
from plotly.colors import sample_colorscale
import pytz
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, precision_score, recall_score, f1_score
from scipy.stats import spearmanr

openai.api_key = userdata.get('OPENAI_API_KEY')

# Retorna un conjunto de datos de entrenamiento o prueba
def generate_set(dataset, set_size, seed=42, balance=False, exclude_set=None):
    random.seed(seed)

    group_size = set_size // 4  # Tamaño grupo para set balanceado
    proportions = df['real_eval'].value_counts(normalize=True)  # Para set no balanceado
    samples_per_class = (proportions * set_size).round().astype(int)

    # Crear set excluyendo las filas de exclude_set
    final_set = dataset
    if exclude_set is not None:
        final_set = dataset[~dataset['row'].isin(exclude_set['row'])]

    if balance:
        final_set = final_set.groupby('real_eval', group_keys=False)[df.columns.tolist()].apply(
            lambda x: x.sample(group_size, random_state=random.randint(0, 100000))
        )
    else:
        final_set = final_set.groupby('real_eval', group_keys=False)[df.columns.tolist()].apply(
            lambda x: x.sample(samples_per_class[x.name], random_state=random.randint(0, 100000))
        )

    return final_set

# Genera las respuestas con ChatGPT
def eval_gpt(df, prompt, criteria, model, temperature):
    df_copy = df.copy()
    api = OpenAIMultiClient(endpoint="chats", data_template={"model": model, "temperature": temperature, "n": 1, "timeout":10}, concurrency=50, wait_interval=1, max_retries=3, retry_max=10, retry_multiplier=1)

    texts = []
    for i, row in df_copy.iterrows():
        text = prompt.format(Question=row['question'], Answer=row['answer'], Context=row['context'])
        texts.append(text)

    answers_gpt = chat_gpt_multiple(api, texts)
    gpt_dicts = extract_dicts(answers_gpt)
    clean_set(df_copy, gpt_dicts, criteria)
    df_dicts = pd.DataFrame(gpt_dicts)
    response_set = pd.concat([df_copy, df_dicts.set_index(df_copy.index)], axis=1)
    return response_set

# Extrae diccionario de salida de las respuestas GPT
def extract_dicts(answers_gpt):
    pattern = r'\{[^{}]+\}'

    gpt_dicts = []
    for answer_gpt in answers_gpt:
        try:
            answer = re.findall(pattern, answer_gpt[0])[0]
            gpt_dicts.append(eval(answer))
        except Exception as e:
            print(f"Error al extraer diccionario. Respuesta GPT: \n{answer_gpt[0]}\n\n")
            gpt_dicts.append(None)

    return gpt_dicts

# Elimina filas del dataset donde hubo errores en la salida GPT
def clean_set(dataset, gpt_dicts, criteria):
    for i in reversed(range(len(gpt_dicts))):
        if gpt_dicts[i] is None:
            gpt_dicts.pop(i)
            dataset.drop(dataset.index[i], inplace=True)
        elif all(key in gpt_dicts[i] for key in criteria) == False:
            print(gpt_dicts[i])
            gpt_dicts.pop(i)
            dataset.drop(dataset.index[i], inplace=True)

# Obtiene los puntajes reales de un dataset
def get_real_scores(dataset):
    return dataset['real_eval'].tolist()

# Prepara el set de entrenamiento y obtiene los parámetros óptimos para disminuir el error
def train(train_set, prompt, criteria, eval_function, model, temperature):
    train_set = train_set.copy()
    res_df = eval_gpt(train_set, prompt, criteria, model, temperature)

    real_scores = get_real_scores(train_set)
    criteria_scores = res_df[criteria].values.tolist()
    params = optimize_params(criteria_scores, real_scores, eval_function)
    return np.round(params, 2)

# Retorna un dataset con el MSE por grupo
def calculate_mse(result_set, normalize):
    if normalize:
        mse_dict = result_set.groupby('dataset')[result_set.columns.tolist()].apply(lambda x: mean_squared_error(x['real_eval']/3, x['gpt_eval']/3)).to_dict()
        overall_mse = mean_squared_error(result_set['real_eval']/3, result_set['gpt_eval']/3)
    else:
        mse_dict = result_set.groupby('dataset')[result_set.columns.tolist()].apply(lambda x: mean_squared_error(x['real_eval'], x['gpt_eval'])).to_dict()
        overall_mse = mean_squared_error(result_set['real_eval'], result_set['gpt_eval'])

    mse_dict['All'] = overall_mse
    return mse_dict

# Muestra un gráfico con los puntajes obtenidos para cada pregunta ordenadas por puntaje real (0-3)
def show_distribution(full_df):
    rep = full_df['repetition'].nunique()
    full_df = full_df.drop(columns=['params'])

    info_cols = ['row', 'dataset', 'question', 'answer', 'context']
    extra_cols = full_df.columns.difference(['question', 'answer', 'context', 'real_eval', 'row', 'dataset', 'repetition', 'gpt_eval', 'score']).tolist()
    for col in info_cols + extra_cols:
        if full_df[col].dtype == 'object':
            full_df[col] = full_df[col].str.wrap(80).apply(lambda x: x.replace('\n', '<br>'))

    full_df = full_df.pivot(index=['question', 'answer', 'context', 'real_eval', 'row', 'dataset'], columns='repetition', values=['gpt_eval', 'score'] + extra_cols)
    full_df = full_df.sort_values('real_eval').reset_index()
    full_df.loc[full_df['dataset'] == 'C1-OscarBadAnswers20', 'dataset'] = 'C1-BadAnswers'

    value_counts = full_df['real_eval'].value_counts()
    n = np.array([value_counts.get(i, 0) for i in range(4)])
    x_pos = [x + (y+1)/(n[x]+1) for x in range(4) for y in range(n[x])]

    def plot(col, title):
        dev = full_df[col].apply(lambda row: row.std(ddof=0), axis=1).values.tolist()
        mean = full_df[col].apply(lambda row: round(row.mean(), 2), axis=1).values.tolist()

        fig = go.Figure()

        if col == 'gpt_eval':
            for i in range(4):
                fig.add_trace(go.Scatter(
                    x=[i, i + 1],
                    y=[i, i],
                    mode='lines',
                    line=dict(color='red', width=1),
                    showlegend=False,
                    hoverinfo='none'
                ))

        # Información extra
        if rep == 1:
            template_cols = info_cols + [col] + extra_cols
            customdata = list(zip(*[full_df[col] for col in info_cols], mean, *[full_df[col][1] for col in extra_cols]))
        else:
            template_cols = info_cols + [col]
            customdata = list(zip(*[full_df[col] for col in info_cols], mean))

        template = ''
        for i, x in enumerate(template_cols):
            template += f'<b>{x}:</b> %{{customdata[{i}]}}<br>'
        template += '<extra></extra>'

        # Colores por clase (dataset)
        colors = sample_colorscale(px.colors.qualitative.Plotly, [i / 3 for i in range(4)])

        # Añadir los puntos
        for n, cls in enumerate(sorted(full_df['dataset'].unique())):
            indices = full_df.index[full_df['dataset'] == cls].tolist()
            x_filtered = [x_pos[i] for i in indices]
            y_filtered = [mean[i] for i in indices]
            dev_filtered = [dev[i] for i in indices]
            customdata_filtered = [customdata[i] for i in indices]

            # Color barra de error
            color_hex = px.colors.qualitative.Plotly[n]
            r = int(color_hex[1:3], 16)
            g = int(color_hex[3:5], 16)
            b = int(color_hex[5:7], 16)
            alpha = 0.5
            color_rgba = f"rgba({r}, {g}, {b}, {alpha})"

            fig.add_trace(go.Scatter(
                x=x_filtered,
                y=y_filtered,
                mode='markers',
                marker=dict(size=8, color=color_hex),
                error_y=dict(type='data', array=dev_filtered, visible=True, thickness=2, width=4, color=color_rgba),
                customdata=customdata_filtered,
                hovertemplate=template,
                name=cls
            ))

        # Configurar el layout
        y_range = (0, 3) if col == 'gpt_eval' else (0, 10)
        fig.update_layout(
            title=title,
            title_x=0.5,
            xaxis_title='Real Eval',
            yaxis_title='GPT Eval',
            xaxis=dict(tickvals=[0, 1, 2, 3]),
            legend_title_text='Dataset',
            template='plotly_white',
            showlegend=True,
            width=1200,
            height=700,
            hoverlabel=dict(
                bgcolor="green",
                font_size=12
            )
        )

        # Mostrar el gráfico
        fig.show()
        print()

    plot('gpt_eval', 'Distribution of scores obtained by the model')
    #plot('score', 'Distribución de puntajes GPT sin normalizar')

# Exporta las salidas GPT a un archivo EXCEL
def export_responses(response_set, prompt, repetitions, set_size, seed, model, temperature, balance_set, repeat_set):
    dir = "Responses"
    if not os.path.exists(dir):
        os.makedirs(dir)

    timezone = pytz.timezone('America/Santiago')
    date = datetime.now(timezone)
    formatted_date = date.strftime('%Y%m%d-%H%M%S')
    filename = f'{dir}/{formatted_date}.xlsx'

    metadata = {
        'prompt': prompt,
        'repetitions': repetitions,
        'set_size': set_size,
        'seed': seed,
        'model': model,
        'temperature': temperature,
        'repeat_set': repeat_set,
        'balance_set': balance_set
    }
    md_set = pd.DataFrame.from_dict(metadata, orient='index')

    with pd.ExcelWriter(filename) as writer:
        response_set.to_excel(writer, sheet_name='Responses', index=False)
        md_set.to_excel(writer, sheet_name='Metadata', index=True, header=False)
    return filename

# Genera salidas GPT para un conjunto de datos
def generate_responses(dataset, prompts, repetitions, set_size=100, seed=42, model="gpt-4o-mini", temperature=0.1, balance_set=False, repeat_set=False):
    sets = []
    for i in range(repetitions):
        if repeat_set:
            sets.append(generate_set(dataset, set_size, seed, balance=False, exclude_set=None))
        else:
            sets.append(generate_set(dataset, set_size, seed + i, balance=False, exclude_set=None))

    filenames = []
    for i, prompt_data in enumerate(prompts):
        prompt = prompt_data.prompt
        criteria = prompt_data.criteria
        response_set = pd.DataFrame()

        for j in range(repetitions):
            print(f"Generando respuestas para Prompt {i+1} / Conjunto {j+1}")
            rep_set = eval_gpt(sets[j], prompt, criteria, model, temperature)
            rep_set['repetition'] = j+1

            response_set = pd.concat([response_set, rep_set], ignore_index=True)
            print()

        prompt_structure = json.dumps(prompt_data.base_structure())
        filename = export_responses(response_set, prompt_structure, repetitions, set_size, seed, model, temperature, balance_set, repeat_set)
        filenames.append(filename)

# Lee y muestra resultados de un archivo EXCEL
def read_evals(filenames, normalize=False):
    df_mean_all = pd.DataFrame()
    df_std_all = pd.DataFrame()

    for filename in filenames:
        full_df = pd.read_excel(filename, sheet_name='Evaluation')
        md = pd.read_excel(filename, sheet_name='Metadata (res)', header=None, index_col=0).T.reset_index(drop=True)
        df = full_df.copy()
        df = df[['repetition', 'dataset', 'real_eval', 'gpt_eval']]
        df['rounded_gpt_eval'] = round(df['gpt_eval']).astype(int)

        mp_stats = []
        for repetition in df['repetition'].unique():
            df_rep = df[df['repetition'] == repetition]

            # Métricas de regresión
            if normalize:
                micro_mse = mean_squared_error(df_rep['real_eval']/3, df_rep['gpt_eval']/3)
                mse_per_set = df_rep.groupby('dataset')[df_rep.columns.tolist()].apply(lambda x: mean_squared_error(x['real_eval']/3, x['gpt_eval']/3)).to_dict()
                mae = mean_absolute_error(df_rep['real_eval']/3, df_rep['gpt_eval']/3)
            else:
                micro_mse = mean_squared_error(df_rep['real_eval'], df_rep['gpt_eval'])
                mse_per_set = df_rep.groupby('dataset')[df_rep.columns.tolist()].apply(lambda x: mean_squared_error(x['real_eval'], x['gpt_eval'])).to_dict()
                mae = mean_absolute_error(df_rep['real_eval'], df_rep['gpt_eval'])

            mse_per_set = {'mse_' + k: v for k, v in mse_per_set.items()}
            mse_stats = {
                'micro_mse': micro_mse,
            }
            mse_stats.update(mse_per_set)

            # MSE por clase (para calcular el macro)
            mse_per_class = {}
            for cls in sorted(df_rep['real_eval'].unique()):
                y_true = df_rep.loc[df_rep['real_eval'] == cls, 'real_eval']
                y_pred = df_rep.loc[df_rep['real_eval'] == cls, 'gpt_eval']
                if len(y_true) > 0:
                    mse_per_class[cls] = mean_squared_error(y_true, y_pred)

            macro_mse = np.mean(list(mse_per_class.values()))

            spearman, p_value = spearmanr(df_rep['real_eval'], df_rep['gpt_eval'])
            r2 = r2_score(df_rep['real_eval'], df_rep['gpt_eval'])

            # Métricas de clasificación: Se calculan con puntaje GPT redondeado (0-3)
            accuracy = accuracy_score(df_rep['real_eval'], df_rep['rounded_gpt_eval'])
            precision = precision_score(df_rep['real_eval'], df_rep['rounded_gpt_eval'], average='weighted', zero_division=0)
            recall = recall_score(df_rep['real_eval'], df_rep['rounded_gpt_eval'], average='weighted', zero_division=0)
            f1 = f1_score(df_rep['real_eval'], df_rep['rounded_gpt_eval'], average='weighted', zero_division=0)

            stats = {
                'spearman': spearman,
                'macro_mse': macro_mse
            }
            stats.update(mse_stats)
            stats.update({
                'mae': mae,
                'r2': r2,
                'accuracy': accuracy,
                'precision': precision,
                'recall': recall,
                'f1': f1
            })
            mp_stats.append(stats)

        mp_stats = pd.DataFrame(mp_stats)
        df_mean = mp_stats.apply(['mean'])
        df_std = mp_stats.apply([np.std])

        df_mean.insert(0, 'prompt', md['prompt'].iloc[0])
        df_std.insert(0, 'prompt', md['prompt'].iloc[0])
        df_mean_all = pd.concat([df_mean_all, df_mean], ignore_index=True)
        df_std_all = pd.concat([df_std_all, df_std], ignore_index=True)

    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', None)

    print("\nTabla Promedios")
    display(df_mean_all)
    print("\nTabla Desviación estándar")
    display(df_std_all)
    pd.reset_option('^display.', silent=True)

    repeat_test_set = md['repeat_set'].iloc[0]
    repetitions = md['repetitions'].iloc[0]
    if len(filenames) == 1 and (repeat_test_set or repetitions == 1):
        print()
        show_distribution(full_df)

# Exportar archivo de evaluaciones
def export_eval(filename, eval_set, res_md, eval_function, eval_params, train_set_size, seed, model, temperature):
    dir = "Evals"
    if not os.path.exists(dir):
        os.makedirs(dir)

    timezone = pytz.timezone('America/Santiago')
    date = datetime.now(timezone)
    formatted_date = date.strftime('%Y%m%d-%H%M%S')

    filename = os.path.splitext(os.path.basename(filename))[0]

    if eval_params is None:
        filename = f'{dir}/T_{filename}_eval.xlsx'
    else:
        filename = f'{dir}/{filename}_eval.xlsx'

    eval_md = {
        'eval_function': eval_function,
        'eval_params': eval_params,
        'train_set_size': train_set_size,
        'seed': seed,
        'model': model,
        'temperature': temperature
    }
    eval_md = pd.DataFrame.from_dict(eval_md, orient='index')

    with pd.ExcelWriter(filename) as writer:
        eval_set.to_excel(writer, sheet_name='Evaluation', index=False)
        eval_md.to_excel(writer, sheet_name='Metadata (eval)', index=True, header=False)
        res_md.to_excel(writer, sheet_name='Metadata (res)', index=True, header=False)
    return filename

# Evalúa en base a las salidas de GPT
def evaluate(filename, dataset, eval_function, eval_params=None, train_set_size=100, seed=42, model="gpt-4o-mini", temperature=0.1, prompt_folder="GPTEvaluator/Experiments/Miniprompts_v2"):
    res_df = pd.read_excel(filename, sheet_name='Responses')

    res_md = pd.read_excel(filename, sheet_name='Metadata', header=None, index_col=0)
    prompt_dict = json.loads(res_md.T.reset_index(drop=True)['prompt'][0])
    prompt = generate_prompts(prompt_dict, prompt_folder, visualize=False)[0]

    # Dataset final de evaluaciones
    eval_set = pd.DataFrame()

    for repetition in res_df['repetition'].unique():
        rep_df = res_df[res_df['repetition'] == repetition]
        test_set = rep_df[rep_df['row'].isin(dataset['row'])].copy()

        # ENTRENAMIENTO
        rep_params = eval_params
        if eval_params is None:
            print(f"Evaluando conjunto de prueba {repetition} con AJUSTE")
            train_set = generate_set(dataset, train_set_size, int(seed + repetition), balance=False, exclude_set=test_set)
            rep_params = train(train_set, prompt.prompt, prompt.criteria, eval_function, model, temperature)
            str_params = "[" + ", ".join("{:.2f}".format(param) for param in rep_params) + "]"
            print()
            print(f"Parámetros obtenidos: {str_params}")

        real_scores = rep_df['real_eval'].tolist()
        criteria_scores = rep_df[prompt.criteria].values.tolist()

        pred_scores = convert_gpt_scores(criteria_scores, real_scores, eval_function, rep_params)
        test_set['gpt_eval'] = pred_scores
        test_set['params'] = ", ".join("{:.2f}".format(param) for param in rep_params)

        eval_set = pd.concat([eval_set, test_set], ignore_index=True)

    if eval_params is None: print()
    return export_eval(filename, eval_set, res_md, eval_function, eval_params, train_set_size, seed, model, temperature)

def evaluate_mul(filenames, dataset, eval_function, eval_params=None, train_set_size=100, seed=42, model="gpt-4o-mini", temperature=0.1, prompt_folder="GPTEvaluator/Experiments/Miniprompts_v2"):
    eval_files = []
    for filename in filenames:
        print(f"EVALUANDO {filename}")
        eval_file = evaluate(filename, dataset, eval_function, eval_params, train_set_size, seed, model, temperature, prompt_folder)
        eval_files.append(eval_file)

    if len(eval_files) > 0:
        read_evals(eval_files)

# Espacio para experimentos

In [ ]:
column_data = {
    "context": "Contexto simple",
    "question": "Pregunta",
    "answer": "Respuesta",
    "real_eval": "Promedio Redondeado",
    "dataset": "DataSet"
}

df = load_dataset("datasets_v2.xlsx", "AllDatasets (1dif)", column_data)

,context,question,answer,real_eval,dataset,row
0,BFS es preferible en problemas que requieren e...,¿En qué tipo de problemas una búsqueda BFS pod...,El bfs es mucho mas util en ejecuciones cortas...,2,C3-Sample100,2
1,BFS es preferible en problemas que requieren e...,¿En qué tipo de problemas una búsqueda BFS pod...,en una situación de resolución de un problema ...,2,C3-Sample100,3
2,BFS es preferible en problemas que requieren e...,¿En qué tipo de problemas una búsqueda BFS pod...,en problemas donde pidan obtener el camino de ...,3,C3-Sample100,4
3,BFS es preferible en problemas que requieren e...,¿En qué tipo de problemas una búsqueda BFS pod...,en problemas de grafos no ponderados ya que no...,3,C3-Sample100,5
4,"BFS explora nodos nivel por nivel, visitando p...",¿Cuál es la principal diferencia entre búsqued...,La búsqueda por anchura tiene un procedimiento...,3,C3-Sample100,6



real_eval
3    127
2     75
0     52
1     36
Name: count, dtype: int64

dataset
C2-Nan                  92
C3-Sample100            91
C2-Sample100            90
C1-OscarBadAnswers20    17
Name: count, dtype: int64


In [ ]:
prompt_data = {
    "context": "context.txt",
    "examples_basic": "examples_4G4B_basic.txt",
    "question": "question.txt",
    "answer": "answer.txt",
    "instructions": {
        "score": "score_single.txt",
        "feedback": "*"
    }
}

prompt_folder = "GPTEvaluator/Experiments/Miniprompts_v2"

prompts = generate_prompts(prompt_data, prompt_folder)

**Knowledge:** {Context}

### Examples
**Question**: ¿Cuando se recomienda utilizar arreglos en vez de listas enlazadas? Haga referencia a complejidades temporales en su explicación.
**Student's Answer**: Se recomienta utilizar arreglos cuando se quiere acceder a elementos en posiciones específicas. Acceder a posiciones en un arreglo tiene complejidad temporal O(1), mientras que en una lista enlazada la complejidad es O(n).
**Score**: 10

**Question**: ¿Cuál es la complejidad temporal del peor caso para la operación de búsqueda en una tabla hash y por qué? Describe las condiciones que debe tener la tabla para encontrarse en este peor caso.
**Student's Answer**: La complejidad del peor caso es O(n). Esta puede ocurrir cuando todos los datos de la tabla se encuentran contiguos en el arreglo, y cuando se busca una clave, esta búsqueda hace colisión con todos los datos que estaban almacenados.
**Score**: 10

**Question**: ¿Cómo se podría implementar un historial de navegación web usando do

In [ ]:
generate_responses(df, prompts, repetitions=10, set_size=100, seed=42, model="gpt-4o-mini-2024-07-18", temperature=0.1, balance_set=False, repeat_set=False)

Generando respuestas para Prompt 1 / Conjunto 1
41-29-5-14-49-39-43-47-21-44-37-28-13-17-6-26-22-33-25-9-19-7-32-10-45-40-48-12-35-30-0-18-20-3-4-31-16-42-1-11-34-23-24-8-36-27-2-46-38-15-59-50-80-63-70-56-57-52-62-58-72-60-69-61-74-77-76-75-78-54-82-96-53-55-68-71-88-79-73-90-93-99-51-85-81-66-98-92-67-84-91-87-65-83-94-97-95-89-86-64-
Generando respuestas para Prompt 1 / Conjunto 2
47-30-28-17-5-26-7-13-14-45-6-10-49-33-9-34-32-37-35-11-8-43-42-31-38-40-41-46-2-39-22-24-29-21-0-12-3-4-25-20-44-1-48-27-16-36-19-18-15-23-66-56-51-58-71-67-55-61-57-53-77-72-64-89-75-79-54-68-87-69-81-50-91-60-59-76-90-63-62-70-65-93-86-94-83-95-97-82-80-74-88-84-52-85-73-92-78-98-99-96-
Generando respuestas para Prompt 1 / Conjunto 3
10-2-12-26-19-47-30-4-15-25-5-1-35-22-11-36-16-8-20-32-38-3-21-40-43-23-6-29-18-44-39-14-37-13-17-24-42-49-46-41-28-7-45-0-27-9-31-33-48-34-56-52-57-53-50-61-66-83-78-73-55-51-54-68-91-63-60-79-59-80-85-87-76-72-58-71-74-97-77-82-88-75-86-98-93-65-84-81-69-92-94-64-99-62-95

ERROR:openai_multi_client:Error processing Payload(endpoint='chats', data={'model': 'gpt-4o-mini-2024-07-18', 'temperature': 0.1, 'n': 1, 'timeout': 10, 'messages': [{'role': 'user', 'content': '**Knowledge:** Modela usuarios como nodos y conexiones como aristas. Usa BFS para explorar desde A, nivel por nivel, hasta encontrar B, contando los niveles como el grado de separación. Retorna el nivel en que se encuentra B.\n\n### Examples\n**Question**: ¿Cuando se recomienda utilizar arreglos en vez de listas enlazadas? Haga referencia a complejidades temporales en su explicación.\n**Student\'s Answer**: Se recomienta utilizar arreglos cuando se quiere acceder a elementos en posiciones específicas. Acceder a posiciones en un arreglo tiene complejidad temporal O(1), mientras que en una lista enlazada la complejidad es O(n).\n**Score**: 10\n\n**Question**: ¿Cuál es la complejidad temporal del peor caso para la operación de búsqueda en una tabla hash y por qué? Describe las condiciones que debe

5-1-35-3-44-2-38-33-45-25-26-42-48-39-0-24-47-8-46-6-30-27-14-19-7-49-23-43-31-32-28-40-36-34-22-41-10-13-15-9-18-12-21-20-11-37-50-51-62-29-68-59-53-56-64-57-70-60-76-84-67-82-80-55-65-79-61-88-78-4-69-75-71-73-58-72-89-86-54-97-77-93-91-66-81-74-85-52-96-95-87-92-94-98-90-83-63-99-
Generando respuestas para Prompt 1 / Conjunto 6
34-8-2-15-44-3-28-26-17-16-47-31-27-9-0-25-11-24-4-39-21-22-38-7-18-43-45-33-23-32-14-30-12-42-48-10-6-37-36-35-1-29-41-20-5-49-13-40-19-51-76-53-50-71-67-72-75-52-69-59-55-73-60-86-79-90-83-65-78-85-63-88-57-62-61-64-77-70-58-80-81-56-91-84-93-66-96-82-89-46-95-74-92-87-97-98-99-68-54-94-
Generando respuestas para Prompt 1 / Conjunto 7
1-11-40-46-42-17-45-2-23-33-16-18-38-21-26-9-19-7-0-8-20-29-22-15-3-12-10-35-13-25-4-34-31-6-24-27-49-36-44-32-47-5-43-30-14-41-58-71-62-57-68-54-76-28-80-67-65-63-55-82-81-51-64-72-86-53-70-77-84-90-56-89-37-87-50-92-78-93-88-60-48-59-91-75-95-85-73-94-61-39-66-69-83-97-52-96-79-74-99-98-
Generando respuestas para Prompt 1 / 

ERROR:openai_multi_client:Error processing Payload(endpoint='chats', data={'model': 'gpt-4o-mini-2024-07-18', 'temperature': 0.1, 'n': 1, 'timeout': 10, 'messages': [{'role': 'user', 'content': '**Knowledge:** Tener nodos con múltiples claves en árboles B/B+ permite almacenar más información en cada nodo, reduciendo la altura del árbol y mejorando la eficiencia en las operaciones.\n\n### Examples\n**Question**: ¿Cuando se recomienda utilizar arreglos en vez de listas enlazadas? Haga referencia a complejidades temporales en su explicación.\n**Student\'s Answer**: Se recomienta utilizar arreglos cuando se quiere acceder a elementos en posiciones específicas. Acceder a posiciones en un arreglo tiene complejidad temporal O(1), mientras que en una lista enlazada la complejidad es O(n).\n**Score**: 10\n\n**Question**: ¿Cuál es la complejidad temporal del peor caso para la operación de búsqueda en una tabla hash y por qué? Describe las condiciones que debe tener la tabla para encontrarse en e

35-1-46-8-2-17-39-24-40-6-15-5-9-20-28-43-0-41-10-36-47-45-23-16-33-19-22-30-21-4-18-42-13-49-14-32-31-34-37-48-12-29-38-11-26-3-25-51-50-79-73-77-52-66-55-59-63-53-76-78-54-57-64-60-67-71-61-83-80-70-56-27-86-88-7-65-89-74-84-62-72-68-44-93-87-92-94-82-90-91-97-69-75-85-98-96-99-81-58-95-
Generando respuestas para Prompt 1 / Conjunto 10
37-21-2-14-8-45-5-28-36-1-44-25-34-47-24-12-6-15-42-7-4-49-18-30-23-29-13-41-20-9-35-11-0-43-17-33-38-16-31-22-32-27-46-40-39-19-3-10-57-70-50-59-58-52-68-53-76-65-67-72-63-69-54-82-55-75-81-60-77-61-79-64-80-62-66-71-85-96-83-84-78-56-73-74-87-90-86-94-95-88-48-93-89-26-98-91-92-51-99-97-
Generando respuestas para Prompt 2 / Conjunto 1
14-29-49-7-24-13-25-22-35-44-27-1-37-21-6-39-17-9-42-38-46-8-48-16-5-2-33-18-28-45-43-12-23-40-4-15-10-30-20-31-0-26-36-11-3-34-19-32-41-50-47-80-59-63-51-66-56-69-61-52-60-79-72-85-70-57-58-71-91-62-75-78-68-88-64-77-76-65-82-89-73-87-67-95-86-53-81-92-84-74-96-54-98-97-94-93-90-99-83-55-
Generando respuestas para Prom

In [ ]:
'''
dir = 'Responses/'
filenames = ['analysis_minimal.xlsx', 'analysis_normal.xlsx', 'analysis_full.xlsx']
filenames = [dir + f for f in filenames]
'''

import os

def get_files_starting_with(directory, prefix):
    """Devuelve una lista con los archivos en 'directory' que comienzan con 'prefix'."""
    return [
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if f.startswith(prefix)
    ]

# Ejemplo de uso:
dir_path = "Responses/"
filenames = get_files_starting_with(dir_path, "feedback_normal")

evaluate_mul(filenames, df, eval_function="map2-simple", eval_params=None, train_set_size=100, seed=42, model="gpt-4o-mini-2024-07-18", temperature=0.1)

EVALUANDO Responses/feedback_normal.xlsx
Evaluando conjunto de prueba 1 con AJUSTE
41-9-15-46-40-33-43-13-6-1-5-12-37-4-27-14-2-36-0-49-16-29-3-30-42-21-39-25-35-48-18-26-20-10-17-44-11-38-8-45-50-24-28-51-32-7-55-69-47-34-64-72-54-58-78-61-70-59-62-75-60-67-52-74-77-53-71-57-63-80-83-68-76-22-19-82-85-81-79-73-31-91-89-56-94-66-84-92-98-88-99-23-86-90-97-87-65-93-96-95-
Parámetros obtenidos: [1.00, 1.53, 9.02]
Evaluando conjunto de prueba 2 con AJUSTE
7-26-37-20-22-1-35-10-13-29-6-31-48-41-36-27-40-17-2-14-15-49-3-19-46-42-24-18-5-9-47-16-32-8-28-30-44-4-25-38-11-33-12-34-39-0-57-23-21-53-72-60-52-50-45-85-43-86-75-59-56-64-76-67-61-58-88-66-69-63-81-80-71-55-68-77-65-84-54-70-97-83-79-82-96-73-87-51-99-93-90-62-95-91-92-94-74-78-98-89-
Parámetros obtenidos: [1.00, 1.39, 9.31]
Evaluando conjunto de prueba 3 con AJUSTE
13-49-30-33-45-36-43-40-18-12-5-0-46-2-42-6-39-23-35-48-11-34-27-32-22-26-10-8-25-19-4-41-21-24-20-1-3-44-16-9-37-14-50-15-31-29-28-61-52-57-68-17-56-75-47-72-51-58-7-70

,prompt,spearman,macro_mse,micro_mse,mse_C1-OscarBadAnswers20,mse_C2-Nan,mse_C2-Sample100,mse_C3-Sample100,mae,r2,accuracy,precision,recall,f1
0,"{""context"": ""context.txt"", ""examples_basic"": ""examples_4G4B_basic.txt"", ""question"": ""question.txt"", ""answer"": ""answer.txt"", ""instructions"": {""score"": ""score_single.txt"", ""feedback"": ""feedback_normal.txt""}}",0.685149,0.679315,0.618668,0.584859,0.779575,0.54418,0.55455,0.586506,0.516061,0.512,0.594367,0.512,0.51526



Tabla Desviación estándar


,prompt,spearman,macro_mse,micro_mse,mse_C1-OscarBadAnswers20,mse_C2-Nan,mse_C2-Sample100,mse_C3-Sample100,mae,r2,accuracy,precision,recall,f1
0,"{""context"": ""context.txt"", ""examples_basic"": ""examples_4G4B_basic.txt"", ""question"": ""question.txt"", ""answer"": ""answer.txt"", ""instructions"": {""score"": ""score_single.txt"", ""feedback"": ""feedback_normal.txt""}}",0.028883,0.095554,0.077887,0.208679,0.085526,0.130652,0.14493,0.032533,0.060925,0.029933,0.034611,0.029933,0.025547


In [ ]:
dir_path = "Evals/"
filenames = get_files_starting_with(dir_path, "T_")

read_evals(filenames, normalize=True)


Tabla Promedios


,prompt,spearman,macro_mse,micro_mse,mse_C1-OscarBadAnswers20,mse_C2-Nan,mse_C2-Sample100,mse_C3-Sample100,mae,r2,accuracy,precision,recall,f1
0,"{""context"": ""context.txt"", ""examples_basic"": ""examples_4G4B_basic.txt"", ""question"": ""question.txt"", ""answer"": ""answer.txt"", ""instructions"": {""score"": ""score_single.txt"", ""feedback"": ""feedback_normal.txt""}}",0.685149,0.679315,0.068741,0.064984,0.086619,0.060464,0.061617,0.195502,0.516061,0.512000,0.594367,0.512000,0.515260
1,"{""context"": ""context.txt"", ""examples_basic"": ""examples_4G4B_basic.txt"", ""question"": ""question.txt"", ""answer"": ""answer.txt"", ""instructions"": {""score"": ""score_single.txt""}}",0.718628,0.754334,0.069140,0.087455,0.081977,0.056064,0.068586,0.176548,0.513253,0.553000,0.612237,0.553000,0.571789
2,"{""context"": ""context.txt"", ""examples_basic"": ""examples_4G4B_basic.txt"", ""question"": ""question.txt"", ""answer"": ""answer.txt"", ""instructions"": {""analysis"": ""analysis_normal.txt"", ""score"": ""score_single.txt""}}",0.657351,0.716530,0.073802,0.114330,0.085529,0.066083,0.063866,0.198017,0.480915,0.524576,0.642908,0.524576,0.542719
3,"{""context"": ""context.txt"", ""question"": ""question.txt"", ""answer"": ""answer.txt"", ""instructions"": {""score"": ""score_single.txt""}}",0.716124,0.645303,0.062182,0.052418,0.069108,0.055138,0.064517,0.182267,0.562235,0.564000,0.643373,0.564000,0.578110
4,"{""question"": ""question.txt"", ""answer"": ""answer.txt"", ""instructions"": {""score"": ""score_single.txt""}}",0.684908,0.672856,0.067666,0.087769,0.072107,0.062722,0.064555,0.190148,0.523628,0.558000,0.648960,0.558000,0.574952



Tabla Desviación estándar


,prompt,spearman,macro_mse,micro_mse,mse_C1-OscarBadAnswers20,mse_C2-Nan,mse_C2-Sample100,mse_C3-Sample100,mae,r2,accuracy,precision,recall,f1
0,"{""context"": ""context.txt"", ""examples_basic"": ""examples_4G4B_basic.txt"", ""question"": ""question.txt"", ""answer"": ""answer.txt"", ""instructions"": {""score"": ""score_single.txt"", ""feedback"": ""feedback_normal.txt""}}",0.028883,0.095554,0.008654,0.023187,0.009503,0.014517,0.016103,0.010844,0.060925,0.029933,0.034611,0.029933,0.025547
1,"{""context"": ""context.txt"", ""examples_basic"": ""examples_4G4B_basic.txt"", ""question"": ""question.txt"", ""answer"": ""answer.txt"", ""instructions"": {""score"": ""score_single.txt""}}",0.030832,0.096771,0.009466,0.051681,0.010665,0.015100,0.014272,0.012697,0.066638,0.016155,0.024772,0.016155,0.015357
2,"{""context"": ""context.txt"", ""examples_basic"": ""examples_4G4B_basic.txt"", ""question"": ""question.txt"", ""answer"": ""answer.txt"", ""instructions"": {""analysis"": ""analysis_normal.txt"", ""score"": ""score_single.txt""}}",0.033986,0.074686,0.006750,0.047031,0.012885,0.016051,0.010345,0.009721,0.047884,0.035776,0.028877,0.035776,0.032604
3,"{""context"": ""context.txt"", ""question"": ""question.txt"", ""answer"": ""answer.txt"", ""instructions"": {""score"": ""score_single.txt""}}",0.033035,0.100690,0.008669,0.026269,0.012038,0.012608,0.017096,0.012246,0.061033,0.050438,0.034248,0.050438,0.043113
4,"{""question"": ""question.txt"", ""answer"": ""answer.txt"", ""instructions"": {""score"": ""score_single.txt""}}",0.038006,0.095532,0.009467,0.051607,0.015922,0.014438,0.020879,0.013037,0.066648,0.047497,0.041295,0.047497,0.041234


In [ ]:
!zip -r results.zip Results/

  adding: Results/ (stored 0%)
  adding: Results/analysis_normal.xlsx (deflated 0%)
  adding: Results/feedback_full.xlsx (deflated 0%)
  adding: Results/feedback_full_soft.xlsx (deflated 0%)
  adding: Results/T_examples.xlsx (deflated 2%)
  adding: Results/examples_3G1B.xlsx (deflated 1%)
  adding: Results/feedback_normal.xlsx (deflated 0%)
  adding: Results/knowledge_simple.xlsx (deflated 4%)
  adding: Results/examples_4G0B.xlsx (deflated 2%)
  adding: Results/.ipynb_checkpoints/ (stored 0%)
  adding: Results/examples_1G3B.xlsx (deflated 1%)
  adding: Results/feedback_minimal.xlsx (deflated 0%)
  adding: Results/T_analysis.xlsx (deflated 0%)
  adding: Results/analysis_full.xlsx (deflated 0%)
  adding: Results/examples_0G4B.xlsx (deflated 2%)
  adding: Results/examples_4G4B.xlsx (deflated 3%)
  adding: Results/T_qa_score.xlsx (deflated 2%)
  adding: Results/examples_1G1B.xlsx (deflated 1%)
  adding: Results/T_knowledge.xlsx (deflated 1%)
  adding: Results/examples_2G2B.xlsx (deflated 2